<a href="https://colab.research.google.com/github/harry-hoolahan/harry-hoolahan.github.io/blob/main/DCF_Financial_Modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Annual financial statements to Excel

Pulls annual income statement, cash flow and balance sheet data for one ticker from the Alpha Vantage API, keeps 2015-2022, and saves them as separate sheets in an Excel file.

This is the **data-collection step** for a discounted cash flow (DCF) valuation. It does not forecast or value anything. For a working DCF with editable assumptions, see the DCF explorer on the website.

Run the cell below and paste your own Alpha Vantage API key when prompted.

In [ ]:
from google.colab import files
import requests
import pandas as pd
from getpass import getpass

# Alpha Vantage API key (free key: https://www.alphavantage.co/support/#api-key)
# Entered at runtime so it is never saved in the notebook or committed to GitHub.
api_key = getpass('Alpha Vantage API key: ')

ticker = 'GOOGL'

# Define the base URL
base_url = 'https://www.alphavantage.co/query?'

start_year = 2015
end_year = 2022

# Function to fetch and process data
def fetch_and_process_data(financial_statement_function):
    # Define the parameters
    params = {
        "function": financial_statement_function,
        "symbol": ticker,
        "apikey": api_key
    }

    # Send the request
    response = requests.get(base_url, params=params)

    # Load the data into a DataFrame
    data = response.json()

    # Check if there is an error message
    if 'Error Message' in data:
        print(data['Error Message'])
        return

    statements = pd.json_normalize(data, 'annualReports')

    # Set 'fiscalDateEnding' as the index and convert it to datetime
    statements.set_index('fiscalDateEnding', inplace=True)
    statements.index = pd.to_datetime(statements.index)

    # Sort the dataframe to get the years in ascending order
    statements.sort_index(ascending=True, inplace=True)

    # Filter out the years you're interested in
    statements = statements.loc[str(start_year):str(end_year)]

    # Transpose the dataframe
    statements = statements.transpose()

    return statements

# Fetch and print the income statements
income_statements = fetch_and_process_data("INCOME_STATEMENT")

# Fetch and print the cash flow statements
cash_flow_statements = fetch_and_process_data("CASH_FLOW")

# Fetch and print the balance sheets
balance_sheets = fetch_and_process_data("BALANCE_SHEET")

# Create a Pandas Excel writer using XlsxWriter as the engine.
writer = pd.ExcelWriter('financial_statements.xlsx', engine='xlsxwriter')

# Write each DataFrame to a different worksheet.
income_statements.to_excel(writer, sheet_name='Income Statements')
cash_flow_statements.to_excel(writer, sheet_name='Cash Flow Statements')
balance_sheets.to_excel(writer, sheet_name='Balance Sheets')

# Close the Pandas Excel writer and output the Excel file.
writer.close()

# Download the file to your local system
files.download('financial_statements.xlsx')
